Блок 1: Импорт данных и первичная подготовка

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os

path = kagglehub.dataset_download("blastchar/telco-customer-churn")
files = os.listdir(path)
csv_file = [f for f in files if f.endswith('.csv')][0]
full_path = os.path.join(path, csv_file)

df = pd.read_csv(full_path)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
df


Бизнес-контекст и методология:
На данном этапе мы загружаем датасет Telco Customer Churn, который содержит информацию о клиентах телекоммуникационной компании. Ключевые действия:
Загрузка данных - получаем доступ к историческим данным о клиентах, включая их демографические характеристики, подключенные услуги и факт оттока.
Очистка TotalCharges - преобразуем поле общих расходов в числовой формат, заменяя пропуски на 0. Это критично для корректного финансового анализа.
Бинаризация целевой переменной - переводим Churn из категориального формата (Yes/No) в бинарный (1/0) для удобства статистического анализа и моделирования.
Ключевой инсайт: Подготовка данных - это не просто технический шаг, а возможность уже на старте выявить проблемы качества данных (пропуски, некорректные форматы), которые могут исказить дальнейший анализ.

Блок 2: Многомерная визуализация паттернов оттока


In [ ]:
sns.set_theme(style="whitegrid")

fig1 = plt.figure(figsize=(18, 15))

plt.subplot(3, 3, 1)
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen', 'Churn']
corr_matrix = df[numeric_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.title('Correlation Matrix - Numeric Features')

plt.subplot(3, 3, 2)
sns.histplot(data=df, x='tenure', hue='Churn', multiple='stack', bins=30, palette='Set2')
plt.title('Tenure Distribution by Churn')

plt.subplot(3, 3, 3)
sns.histplot(data=df, x='MonthlyCharges', hue='Churn', multiple='stack', bins=30, palette='Set2')
plt.title('Monthly Charges Distribution by Churn')

plt.subplot(3, 3, 4)
sns.histplot(data=df, x='TotalCharges', hue='Churn', multiple='stack', bins=30, palette='Set2')
plt.title('Total Charges Distribution by Churn')

plt.subplot(3, 3, 5)
sns.scatterplot(data=df, x='tenure', y='MonthlyCharges', hue='Churn', alpha=0.6, palette='Set2')
plt.title('Tenure vs Monthly Charges')

plt.subplot(3, 3, 6)
sns.scatterplot(data=df, x='MonthlyCharges', y='TotalCharges', hue='tenure', palette='viridis', alpha=0.6)
plt.title('Monthly vs Total Charges (colored by tenure)')

categorical_cols_first = ['gender', 'Partner', 'Dependents']

for idx, col in enumerate(categorical_cols_first):
    plt.subplot(3, 3, 7 + idx)
    churn_rate = df.groupby(col)['Churn'].mean().sort_values(ascending=False)
    sns.barplot(x=churn_rate.values, y=churn_rate.index, palette='RdYlGn_r')
    plt.title(f'Churn Rate by {col}')
    plt.xlabel('Churn Rate')
    plt.ylabel(col)

plt.tight_layout()
plt.show()

fig2 = plt.figure(figsize=(18, 12))

categorical_cols_second = ['PhoneService', 'InternetService', 'Contract', 
                           'PaperlessBilling', 'PaymentMethod']

for idx, col in enumerate(categorical_cols_second):
    plt.subplot(2, 3, idx + 1)
    churn_rate = df.groupby(col)['Churn'].mean().sort_values(ascending=False)
    sns.barplot(x=churn_rate.values, y=churn_rate.index, palette='RdYlGn_r')
    plt.title(f'Churn Rate by {col}')
    plt.xlabel('Churn Rate')
    plt.ylabel(col)

plt.tight_layout()
plt.show()

print("Correlation with Churn:")
print(df[numeric_cols].corr()['Churn'].sort_values(ascending=False))

print("\n\nChurn Rate by Gender:")
print(df.groupby('gender')['Churn'].mean())

print("\n\nChurn Rate by SeniorCitizen:")
print(df.groupby('SeniorCitizen')['Churn'].mean())

print("\n\nChurn Rate by Partner:")
print(df.groupby('Partner')['Churn'].mean())

print("\n\nChurn Rate by Dependents:")
print(df.groupby('Dependents')['Churn'].mean())

print("\n\nChurn Rate by PhoneService:")
print(df.groupby('PhoneService')['Churn'].mean())

print("\n\nChurn Rate by InternetService:")
print(df.groupby('InternetService')['Churn'].mean())

print("\n\nChurn Rate by Contract:")
print(df.groupby('Contract')['Churn'].mean())

print("\n\nChurn Rate by PaperlessBilling:")
print(df.groupby('PaperlessBilling')['Churn'].mean())

print("\n\nChurn Rate by PaymentMethod:")
print(df.groupby('PaymentMethod')['Churn'].mean())

Бизнес-анализ и стратегические выводы:
1. Tenure (Срок обслуживания) - критический фактор удержания
Мы сегментировали клиентов по сроку обслуживания на 6 групп: 0-3, 4-6, 7-12, 13-24, 25-48, 49+ месяцев.
Ключевой инсайт: Отток максимально высок в первые 3-6 месяцев (период "критической адаптации"), после чего плавно снижается. Это подтверждает гипотезу о том, что первые месяцы - период проверки качества сервиса клиентом.
Бизнес-импликация: Необходима программа onboarding для новых клиентов с активным вовлечением в первые 90 дней.
2. Monthly Charges (Ежемесячный платеж) - ценовая чувствительность
Биннинг по $30, $50, $70, $90, $120 показывает четкую корреляцию: чем выше чек, тем выше отток.
Стратегический вывод: Клиенты с чеком выше $70 находятся в зоне риска. Это может указывать на:
Недостаточную воспринимаемую ценность premium-услуг
Агрессивную конкуренцию в premium-сегменте
Необходимость пересмотра ценовой архитектуры
3. Contract + Payment Method - "Chasers Hypothesis"
Гипотеза "Chasers" (охотники за выгодой) подтверждается:
Month-to-month + Electronic check: 53.73% отток (1850 клиентов)
Month-to-month + другие платежи: 32.64% отток (2025 клиента)
Интерпретация: Клиенты на месячных контрактах с электронными чеками - это "транзакционные" клиенты, которые постоянно ищут лучшую цену и легко переключаются. Электронный чек указывает на низкую лояльность и автоматизированный подход к оплате.
Рекомендация: Для этой сегмента необходима программа лояльности с бонусами за долгосрочную приверженность.
4. Number of Services - парадокс услуг
Анализ показывает нелинейную зависимость: клиенты с 0-2 услугами имеют высокий отток, с 3-5 - умеренный, с 6+ - минимальный.
Инсайт: Cross-selling критичен для удержания. Клиент с 3+ услугами создает "экосистемную зависимость" и switching costs.
5. Fiber Optic vs DSL - технологический разрыв
Сравнение показывает:
Fiber optic + Month-to-month: 54.61% отток
DSL + Month-to-month: 32.22% отток
Парадокс: Несмотря на технологическое превосходство fiber, отток выше. Это может указывать на:
Завышенные ожидания от fiber-сервиса
Проблемы с качеством обслуживания fiber-клиентов
Агрессивный pricing конкурентов в fiber-сегменте
6. Tenure x Services - матрица удержания
Heatmap показывает взаимодействие факторов: новые клиенты (0-3 мес) с малым количеством услуг имеют отток до 56%, тогда как опытные (49+ мес) с 6+ услугами - менее 10%.
Стратегия: Приоритет на cross-selling в первые 6 месяцев.
7. Partner x Dependents - семейный фактор
Клиенты с партнером и иждивенцами показывают значительно меньший отток. Семейные клиенты - стабильный сегмент.

Блок 3: Корреляционный анализ и распределения

In [ ]:
sns.set_theme(style="whitegrid")

fig = plt.figure(figsize=(20, 16))

plt.subplot(3, 3, 1)
df['Tenure_Bin'] = pd.cut(df['tenure'], bins=[-1, 3, 6, 12, 24, 48, 100], labels=['0-3', '4-6', '7-12', '13-24', '25-48', '49+'])
tenure_churn = df.groupby('Tenure_Bin')['Churn'].mean()
sns.barplot(x=tenure_churn.index.astype(str), y=tenure_churn.values, palette='RdYlGn_r')
plt.title('Churn Rate by Tenure Bins')
plt.xlabel('Tenure (months)')
plt.ylabel('Churn Rate')
plt.axhline(y=df['Churn'].mean(), color='red', linestyle='--', label=f'Average: {df["Churn"].mean():.2%}')
plt.legend()

plt.subplot(3, 3, 2)
df['MonthlyCharges_Bin'] = pd.cut(df['MonthlyCharges'], bins=[0, 30, 50, 70, 90, 120], labels=['0-30', '31-50', '51-70', '71-90', '91-120'])
mc_churn = df.groupby('MonthlyCharges_Bin')['Churn'].mean()
sns.barplot(x=mc_churn.index.astype(str), y=mc_churn.values, palette='RdYlGn_r')
plt.title('Churn Rate by Monthly Charges Bins')
plt.xlabel('Monthly Charges ($)')
plt.ylabel('Churn Rate')

plt.subplot(3, 3, 3)
churners = df[df['Churn'] == 1]
contract_payment = churners.groupby(['Contract', 'PaymentMethod']).size().unstack(fill_value=0)
sns.heatmap(contract_payment, annot=True, fmt='d', cmap='YlOrRd')
plt.title('Churners Count by Contract + Payment')
plt.ylabel('Contract')
plt.xlabel('Payment Method')
plt.xticks(rotation=45)

plt.subplot(3, 3, 4)
services_list = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df['Num_Services'] = (df[services_list] == 'Yes').sum(axis=1)
services_churn = df.groupby('Num_Services')['Churn'].mean()
sns.barplot(x=services_churn.index.astype(str), y=services_churn.values, palette='RdYlGn_r')
plt.title('Churn Rate by Number of Services')
plt.xlabel('Number of Services')
plt.ylabel('Churn Rate')

plt.subplot(3, 3, 5)
sns.scatterplot(data=df, x='tenure', y='MonthlyCharges', hue='Churn', alpha=0.4, palette='Set2', s=20)
plt.title('Tenure vs MonthlyCharges (Churn)')
plt.xlabel('Tenure')
plt.ylabel('MonthlyCharges')

plt.subplot(3, 3, 6)
df['Charges_per_Month_Ratio'] = df['TotalCharges'] / (df['tenure'] + 1)
sns.histplot(data=df, x='Charges_per_Month_Ratio', hue='Churn', multiple='stack', bins=50, palette='Set2')
plt.title('Distribution of Avg Monthly Charge')
plt.xlabel('Average Monthly Charge')

plt.subplot(3, 3, 7)
fiber_contract = df[df['InternetService'] == 'Fiber optic'].groupby('Contract')['Churn'].mean()
dsl_contract = df[df['InternetService'] == 'DSL'].groupby('Contract')['Churn'].mean()
x = np.arange(len(fiber_contract.index))
width = 0.35
plt.bar(x - width/2, fiber_contract.values, width, label='Fiber optic', color='orange')
plt.bar(x + width/2, dsl_contract.values, width, label='DSL', color='green')
plt.xticks(x, fiber_contract.index, rotation=45)
plt.title('Churn Rate: Fiber vs DSL by Contract')
plt.xlabel('Contract Type')
plt.ylabel('Churn Rate')
plt.legend()

plt.subplot(3, 3, 8)
tenure_services = df.groupby(['Tenure_Bin', 'Num_Services'])['Churn'].mean().unstack()
sns.heatmap(tenure_services, annot=True, fmt='.2%', cmap='YlOrRd')
plt.title('Churn Rate: Tenure x Services')
plt.xlabel('Num Services')
plt.ylabel('Tenure Bin')

plt.subplot(3, 3, 9)
partner_dependents = df.groupby(['Partner', 'Dependents'])['Churn'].mean().unstack()
sns.heatmap(partner_dependents, annot=True, fmt='.2%', cmap='YlOrRd')
plt.title('Churn Rate: Partner x Dependents')
plt.xlabel('Dependents')
plt.ylabel('Partner')

plt.tight_layout()
plt.show()

print("=== HYPOTHESIS: CHASERS ===")
print("\nChurn rate for Month-to-month + Electronic check:")
mask_chasers = (df['Contract'] == 'Month-to-month') & (df['PaymentMethod'] == 'Electronic check')
print(f"Count: {mask_chasers.sum()}")
print(f"Churn Rate: {df[mask_chasers]['Churn'].mean():.2%}")

print("\n\nChurn rate for Month-to-month + other payments:")
mask_other = (df['Contract'] == 'Month-to-month') & (df['PaymentMethod'] != 'Electronic check')
print(f"Count: {mask_other.sum()}")
print(f"Churn Rate: {df[mask_other]['Churn'].mean():.2%}")

print("\n\n=== TENURE ANALYSIS ===")
print("\nChurn rate by tenure bins:")
print(tenure_churn)

print("\n\n=== SERVICES ANALYSIS ===")
print("\nChurn rate by number of services:")
print(services_churn)

print("\n\n=== FIBER OPTIC vs DSL ===")
print("\nFiber optic churn by contract:")
print(fiber_contract)
print("\nDSL churn by contract:")
print(dsl_contract)

Аналитические инсайты:
Корреляционный анализ: ключевые зависимости
Tenure (-0.35) - сильнейшая отрицательная корреляция с оттоком. Каждый дополнительный месяц обслуживания снижает вероятность оттока.
MonthlyCharges (0.19) - положительная корреляция: рост чека увеличивает риск оттока, но связь умеренная.
TotalCharges (-0.20) - клиенты с большей исторической ценностью (LTV) менее склонны к оттоку.
SeniorCitizen (0.15) - пожилые клиенты имеют несколько повышенный риск оттока. Требует специального подхода.
Категориальный анализ:
Contract (критический фактор):
Month-to-month: 42.71% отток
One year: 11.27% отток
Two year: 2.83% отток
Вывод: Долгосрочные контракты снижают отток в 15 раз по сравнению с месячными.
PaymentMethod:
Electronic check: 45.29% отток
Mailed check: 19.11%
Bank transfer (auto): 16.71%
Credit card (auto): 15.24%
Инсайт: Автоматические платежи снижают отток в 3 раза. Electronic check - индикатор низкой вовлеченности.
InternetService:
Fiber optic: 41.89% отток
DSL: 18.96%
No: 7.41%
Парадокс: Fiber optic, несмотря на премиальность, показывает в 2 раза выше отток, чем DSL.
Partner & Dependents:
Без партнера: 32.96% vs С партнером: 19.66%
Без иждивенцев: 31.28% vs С иждивенцами: 15.45%
Семейные клиенты в 2 раза стабильнее.

Блок 4: Глубокий анализ сервисов и сегментов

In [ ]:
services_list = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

fig = plt.figure(figsize=(20, 14))

plt.subplot(2, 3, 1)
churn_by_service = []
for service in services_list:
    yes_rate = df[df[service] == 'Yes']['Churn'].mean()
    no_rate = df[df[service] != 'Yes']['Churn'].mean()
    churn_by_service.append({'Service': service, 'Status': 'No/None', 'Churn': no_rate})
    churn_by_service.append({'Service': service, 'Status': 'Yes', 'Churn': yes_rate})

plot_df = pd.DataFrame(churn_by_service)
sns.barplot(data=plot_df, x='Service', y='Churn', hue='Status', palette=['green', 'red'])
plt.title('Churn Rate by Service (No/None vs Yes)')
plt.xticks(rotation=45)
plt.legend(title='Has Service')

plt.subplot(2, 3, 2)
new_customers = df[df['tenure'] <= 6]
services_per_new = (new_customers[services_list] == 'Yes').sum(axis=1)
new_churn_by_services = new_customers.groupby(services_per_new)['Churn'].mean()
sns.barplot(x=new_churn_by_services.index.astype(str), y=new_churn_by_services.values, palette='RdYlGn_r', ax=plt.gca())
plt.title('New Customers (0-6m): Churn by Num Services')
plt.xlabel('Number of Services')
plt.ylabel('Churn Rate')

plt.subplot(2, 3, 3)
experienced = df[(df['tenure'] > 6) & (df['tenure'] <= 24)]
services_per_exp = (experienced[services_list] == 'Yes').sum(axis=1)
exp_churn_by_services = experienced.groupby(services_per_exp)['Churn'].mean()
sns.barplot(x=exp_churn_by_services.index.astype(str), y=exp_churn_by_services.values, palette='RdYlGn_r', ax=plt.gca())
plt.title('Experienced (7-24m): Churn by Num Services')
plt.xlabel('Number of Services')
plt.ylabel('Churn Rate')

plt.subplot(2, 3, 4)
anomaly_segment = df[(df['tenure'] >= 4) & (df['tenure'] <= 6) & (df['Num_Services'] == 4)]
print(f"Anomaly segment size: {len(anomaly_segment)}")
print(f"Anomaly segment churn rate: {anomaly_segment['Churn'].mean():.2%}")
print(f"Overall churn rate: {df['Churn'].mean():.2%}")

anomaly_churn = anomaly_segment['Churn'].mean()
overall_churn = df['Churn'].mean()
n_anomaly = len(anomaly_segment)
z_stat = (anomaly_churn - overall_churn) / np.sqrt(overall_churn * (1 - overall_churn) / n_anomaly)
p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))
print(f"Z-statistic: {z_stat:.2f}")
print(f"P-value: {p_value:.4f}")

segment_comparison = df.groupby(['Tenure_Bin', 'Num_Services'])['Churn'].mean().unstack()
sns.heatmap(segment_comparison.loc[['4-6']], annot=True, fmt='.2%', cmap='YlOrRd', ax=plt.gca())
plt.title('Anomaly Check: 4-6 months tenure')
plt.xlabel('Num Services')
plt.ylabel('Tenure Bin')

plt.subplot(2, 3, 5)
upgraders = df[(df['MonthlyCharges'] >= 31) & (df['MonthlyCharges'] <= 50)]
print(f"\nUpgraders segment (31-50$): {len(upgraders)} clients")
print(f"Upgraders churn rate: {upgraders['Churn'].mean():.2%}")
upgrader_services = (upgraders[services_list] == 'Yes').sum(axis=1)
upgrader_churn_by_services = upgraders.groupby(upgrader_services)['Churn'].mean()
sns.barplot(x=upgrader_churn_by_services.index.astype(str), y=upgrader_churn_by_services.values, palette='RdYlGn_r', ax=plt.gca())
plt.title('Upgraders (31-50$): Churn by Num Services')
plt.xlabel('Number of Services')
plt.ylabel('Churn Rate')

plt.subplot(2, 3, 6)
fiber_premium = df[(df['InternetService'] == 'Fiber optic') & (df['MonthlyCharges'] >= 70)]
print(f"\nFiber premium segment (70+$): {len(fiber_premium)} clients")
print(f"Fiber premium churn rate: {fiber_premium['Churn'].mean():.2%}")
fiber_services = (fiber_premium[services_list] == 'Yes').sum(axis=1)
fiber_churn_by_services = fiber_premium.groupby(fiber_services)['Churn'].mean()
sns.barplot(x=fiber_churn_by_services.index.astype(str), y=fiber_churn_by_services.values, palette='RdYlGn_r', ax=plt.gca())
plt.title('Fiber Premium (70+$): Churn by Num Services')
plt.xlabel('Number of Services')
plt.ylabel('Churn Rate')

plt.tight_layout()
plt.show()

print("\n=== SERVICE ADOPTION PATTERNS ===")
print("\nWhich services do churners have vs non-churners?")
for service in services_list:
    churners_with_service = df[(df['Churn'] == 1) & (df[service] == 'Yes')].shape[0]
    non_churners_with_service = df[(df['Churn'] == 0) & (df[service] == 'Yes')].shape[0]
    total_with_service = churners_with_service + non_churners_with_service
    if total_with_service > 0:
        churner_pct = churners_with_service / total_with_service * 100
        print(f"{service}: {churner_pct:.1f}% of users churned ({churners_with_service}/{total_with_service})")

print("\n\n=== HYPOTHESIS TEST: Services overload for new customers ===")
new_with_0_services = new_customers[new_customers['Num_Services'] == 0]['Churn'].mean()
new_with_1_service = new_customers[new_customers['Num_Services'] == 1]['Churn'].mean()
new_with_2_services = new_customers[new_customers['Num_Services'] == 2]['Churn'].mean()
new_with_3plus = new_customers[new_customers['Num_Services'] >= 3]['Churn'].mean()
print(f"New customers (0-6m) with 0 services: {new_with_0_services:.2%} churn")
print(f"New customers (0-6m) with 1 service: {new_with_1_service:.2%} churn")
print(f"New customers (0-6m) with 2 services: {new_with_2_services:.2%} churn")
print(f"New customers (0-6m) with 3+ services: {new_with_3plus:.2%} churn")

n_0 = new_customers[new_customers['Num_Services'] == 0].shape[0]
n_1 = new_customers[new_customers['Num_Services'] == 1].shape[0]
z_01 = (new_with_0_services - new_with_1_service) / np.sqrt(new_with_0_services * (1 - new_with_0_services) / n_0 + new_with_1_service * (1 - new_with_1_service) / n_1)
p_01 = 2 * (1 - stats.norm.cdf(abs(z_01)))
print(f"\nStatistical test: 0 services vs 1 service (new customers)")
print(f"Z-stat: {z_01:.2f}, P-value: {p_01:.4f}")

Стратегические выводы по сервисам:
1. Service Adoption Patterns - парадокс дополнительных услуг
Анализ показывает интересную динамику:
OnlineSecurity: 14.6% оттока среди пользователей (295/2019)
OnlineBackup: 21.5% (523/2429)
DeviceProtection: 22.5% (545/2422)
TechSupport: 15.2% (310/2044)
StreamingTV: 30.1% (814/2707)
StreamingMovies: 29.9% (818/2732)
Ключевой инсайт: Streaming-сервисы показывают наивысший отток, несмотря на популярность. Это указывает на:
Высокую конкуренцию в streaming-сегменте (Netflix, Disney+ и др.)
Восприятие streaming как "дополнительной опции", а не core-сервиса
Необходимость bundling-стратегии
2. Hypothesis Test: Services Overload для новых клиентов
Шокирующий результат:
0 услуг: 45.54% отток
1 услуга: 61.84% отток
2 услуги: 62.56% отток
3+ услуги: 55.32% отток
Интерпретация: Новые клиенты с 1-2 услугами имеют наивысший отток! Это контринтуитивно.
Гипотеза: Клиенты, подключившие 1-2 услуги в первые 6 месяцев, находятся в "серой зоне":
Они уже инвестировали в экосистему (не 0 услуг)
Но недостаточно глубоко интегрированы (не 3+ услуги)
Испытывают "парadox of choice" - не уверены в правильности выбора
Статистическая значимость: Z-stat: -5.20, P-value: 0.0000 - разница между 0 и 1 услугой статистически значима.
3. Anomaly Segment: 4-6 месяцев + 4 услуги
Сегмент клиентов с tenure 4-6 месяцев и 4 услугами показывает аномально высокий отток.
Бизнес-интерпретация: Это клиенты, которые активно подключали услуги в первые месяцы, но к 4-6 месяцу начали "переоценивать" ценность. Возможные причины:
Over-selling на этапе onboarding
Недостаточная активация услуг (клиент не научился пользоваться)
Billing shock - неожиданный рост счета
4. Upgraders Segment ($31-50)
Клиенты, перешедшие в средний ценовой сегмент. Анализ их паттерна услуг показывает, что cross-selling в этом сегменте критичен.
5. Fiber Premium Segment ($70+)
Premium fiber-клиенты с высоким чеком. Если они имеют 3+ услуги - отток минимален. Если 0-2 услуги - высокий риск churn.
Рекомендация: Для premium-сегмента обязательна программа adoption с персональным менеджером.


Блок 5: Детальный анализ новых клиентов (0-6 месяцев)

In [ ]:
new_customers = df[df['tenure'] <= 6].copy()
new_churned = new_customers[new_customers['Churn'] == 1]
new_retained = new_customers[new_customers['Churn'] == 0]

print(f"Total new customers: {len(new_customers)}")
print(f"Churned: {len(new_churned)} ({len(new_churned)/len(new_customers)*100:.1f}%)")
print(f"Retained: {len(new_retained)} ({len(new_retained)/len(new_customers)*100:.1f}%)")

fig = plt.figure(figsize=(20, 14))

plt.subplot(3, 3, 1)
sns.histplot(data=new_customers, x='MonthlyCharges', hue='Churn', multiple='stack', bins=30, palette='Set2')
plt.title('New Customers: MonthlyCharges Distribution')
plt.xlabel('MonthlyCharges')

plt.subplot(3, 3, 2)
sns.histplot(data=new_customers, x='TotalCharges', hue='Churn', multiple='stack', bins=30, palette='Set2')
plt.title('New Customers: TotalCharges Distribution')
plt.xlabel('TotalCharges')

plt.subplot(3, 3, 3)
sns.histplot(data=new_customers, x='tenure', hue='Churn', multiple='stack', bins=6, palette='Set2')
plt.title('New Customers: Tenure Distribution (0-6m)')
plt.xlabel('Tenure (months)')

plt.subplot(3, 3, 4)
contract_churn_new = new_customers.groupby('Contract')['Churn'].mean()
sns.barplot(x=contract_churn_new.index, y=contract_churn_new.values, palette='RdYlGn_r')
plt.title('New Customers: Churn by Contract')
plt.xlabel('Contract Type')
plt.ylabel('Churn Rate')
plt.xticks(rotation=45)

plt.subplot(3, 3, 5)
payment_churn_new = new_customers.groupby('PaymentMethod')['Churn'].mean()
sns.barplot(x=payment_churn_new.index, y=payment_churn_new.values, palette='RdYlGn_r')
plt.title('New Customers: Churn by PaymentMethod')
plt.xlabel('Payment Method')
plt.ylabel('Churn Rate')
plt.xticks(rotation=45)

plt.subplot(3, 3, 6)
internet_churn_new = new_customers.groupby('InternetService')['Churn'].mean()
sns.barplot(x=internet_churn_new.index, y=internet_churn_new.values, palette='RdYlGn_r')
plt.title('New Customers: Churn by InternetService')
plt.xlabel('Internet Service')
plt.ylabel('Churn Rate')

plt.subplot(3, 3, 7)
services_list = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
service_adoption = []
for service in services_list:
    churned_pct = new_churned[new_churned[service] == 'Yes'].shape[0] / len(new_churned) * 100
    retained_pct = new_retained[new_retained[service] == 'Yes'].shape[0] / len(new_retained) * 100
    service_adoption.append({'Service': service, 'Group': 'Churned', 'Adoption %': churned_pct})
    service_adoption.append({'Service': service, 'Group': 'Retained', 'Adoption %': retained_pct})

plot_df = pd.DataFrame(service_adoption)
sns.barplot(data=plot_df, x='Service', y='Adoption %', hue='Group', palette=['red', 'green'])
plt.title('New Customers: Service Adoption Rate')
plt.xticks(rotation=45)
plt.legend(title='Group')

plt.subplot(3, 3, 8)
gender_churn_new = new_customers.groupby('gender')['Churn'].mean()
sns.barplot(x=gender_churn_new.index, y=gender_churn_new.values, palette='RdYlGn_r')
plt.title('New Customers: Churn by Gender')
plt.xlabel('Gender')
plt.ylabel('Churn Rate')

plt.subplot(3, 3, 9)
partner_churn_new = new_customers.groupby('Partner')['Churn'].mean()
dependents_churn_new = new_customers.groupby('Dependents')['Churn'].mean()
x = np.arange(2)
width = 0.35
plt.bar(x - width/2, partner_churn_new.values, width, label='Partner', color='blue')
plt.bar(x + width/2, dependents_churn_new.values, width, label='Dependents', color='orange')
plt.xticks(x, ['No', 'Yes'])
plt.title('New Customers: Churn by Partner/Dependents')
plt.xlabel('Has Partner/Dependents')
plt.ylabel('Churn Rate')
plt.legend()

plt.tight_layout()
plt.show()

print("\n=== STATISTICAL TESTS FOR NEW CUSTOMERS ===")

from scipy import stats

print("\n1. T-test: MonthlyCharges (Churned vs Retained)")
t_stat, p_value = stats.ttest_ind(new_churned['MonthlyCharges'], new_retained['MonthlyCharges'])
print(f"T-stat: {t_stat:.2f}, P-value: {p_value:.4f}")
print(f"Mean MonthlyCharges - Churned: {new_churned['MonthlyCharges'].mean():.2f}")
print(f"Mean MonthlyCharges - Retained: {new_retained['MonthlyCharges'].mean():.2f}")

print("\n2. T-test: TotalCharges (Churned vs Retained)")
t_stat, p_value = stats.ttest_ind(new_churned['TotalCharges'], new_retained['TotalCharges'])
print(f"T-stat: {t_stat:.2f}, P-value: {p_value:.4f}")
print(f"Mean TotalCharges - Churned: {new_churned['TotalCharges'].mean():.2f}")
print(f"Mean TotalCharges - Retained: {new_retained['TotalCharges'].mean():.2f}")

print("\n3. T-test: tenure (Churned vs Retained)")
t_stat, p_value = stats.ttest_ind(new_churned['tenure'], new_retained['tenure'])
print(f"T-stat: {t_stat:.2f}, P-value: {p_value:.4f}")
print(f"Mean tenure - Churned: {new_churned['tenure'].mean():.2f}")
print(f"Mean tenure - Retained: {new_retained['tenure'].mean():.2f}")

print("\n4. Chi-square: Contract vs Churn")
contingency = pd.crosstab(new_customers['Contract'], new_customers['Churn'])
chi2, p_value, dof, expected = stats.chi2_contingency(contingency)
print(f"Chi2: {chi2:.2f}, P-value: {p_value:.4f}")
print(contingency)

print("\n5. Chi-square: PaymentMethod vs Churn")
contingency = pd.crosstab(new_customers['PaymentMethod'], new_customers['Churn'])
chi2, p_value, dof, expected = stats.chi2_contingency(contingency)
print(f"Chi2: {chi2:.2f}, P-value: {p_value:.4f}")

print("\n6. Chi-square: InternetService vs Churn")
contingency = pd.crosstab(new_customers['InternetService'], new_customers['Churn'])
chi2, p_value, dof, expected = stats.chi2_contingency(contingency)
print(f"Chi2: {chi2:.2f}, P-value: {p_value:.4f}")

print("\n7. Chi-square: gender vs Churn")
contingency = pd.crosstab(new_customers['gender'], new_customers['Churn'])
chi2, p_value, dof, expected = stats.chi2_contingency(contingency)
print(f"Chi2: {chi2:.2f}, P-value: {p_value:.4f}")

print("\n8. Chi-square: Partner vs Churn")
contingency = pd.crosstab(new_customers['Partner'], new_customers['Churn'])
chi2, p_value, dof, expected = stats.chi2_contingency(contingency)
print(f"Chi2: {chi2:.2f}, P-value: {p_value:.4f}")

print("\n9. Chi-square: Dependents vs Churn")
contingency = pd.crosstab(new_customers['Dependents'], new_customers['Churn'])
chi2, p_value, dof, expected = stats.chi2_contingency(contingency)
print(f"Chi2: {chi2:.2f}, P-value: {p_value:.4f}")

print("\n=== INTERACTION ANALYSIS ===")
print("\nChurn rate for new customers with Fiber optic + Month-to-month:")
mask = (new_customers['InternetService'] == 'Fiber optic') & (new_customers['Contract'] == 'Month-to-month')
print(f"Count: {mask.sum()}, Churn Rate: {new_customers[mask]['Churn'].mean():.2%}")

print("\nChurn rate for new customers with Fiber optic + One year:")
mask = (new_customers['InternetService'] == 'Fiber optic') & (new_customers['Contract'] == 'One year')
print(f"Count: {mask.sum()}, Churn Rate: {new_customers[mask]['Churn'].mean():.2%}")

print("\nChurn rate for new customers with DSL + Month-to-month:")
mask = (new_customers['InternetService'] == 'DSL') & (new_customers['Contract'] == 'Month-to-month')
print(f"Count: {mask.sum()}, Churn Rate: {new_customers[mask]['Churn'].mean():.2%}")

print("\nChurn rate for new customers with No internet + Month-to-month:")
mask = (new_customers['InternetService'] == 'No') & (new_customers['Contract'] == 'Month-to-month')
print(f"Count: {mask.sum()}, Churn Rate: {new_customers[mask]['Churn'].mean():.2%}")

print("\n=== MONTHLY CHARGES THRESHOLD ANALYSIS ===")
for threshold in [20, 30, 40, 50, 60, 70, 80]:
    above = new_customers[new_customers['MonthlyCharges'] > threshold]
    below = new_customers[new_customers['MonthlyCharges'] <= threshold]
    print(f"Threshold ${threshold}: Above={len(above)} (churn {above['Churn'].mean():.2%}), Below={len(below)} (churn {below['Churn'].mean():.2%})")

Критические инсайты по новым клиентам:
1. Масштаб проблемы: 52.9% отток новых клиентов
Из 1481 новых клиентов (0-6 месяцев):
Churned: 784 (52.9%)
Retained: 697 (47.1%)
Бизнес-импликация: Каждый второй новый клиент уходит в первые полгода. Это критическая проблема unit economics.
2. Статистические тесты: подтвержденные гипотезы
T-test: MonthlyCharges
T-stat: 15.07, P-value: 0.0000
Churned: $63.64 vs Retained: $44.72
Вывод: Ушедшие клиенты платили на 42% больше. Высокий чек - фактор риска для новых клиентов.
T-test: TotalCharges
T-stat: 3.97, P-value: 0.0001
Churned: $155.01 vs Retained: $128.58
Интерпретация: Несмотря на больший общий spend, churned клиенты не успели накопить достаточную "инвестицию" в сервис.
T-test: tenure
T-stat: -5.29, P-value: 0.0000
Churned: 2.30 мес vs Retained: 2.75 мес
Инсайт: Даже разница в 2 недели (0.45 мес) статистически значима. Каждая неделя удержания важна.
3. Chi-square тесты: категориальные факторы
Contract vs Churn: Chi2: 64.04, P-value: 0.0000
12345
Шокирующий результат: Среди новых клиентов на two-year контракте - 0% оттока! На one-year - 10.3%, на month-to-month - 55.2%.
Рекомендация: Агрессивный upsell на долгосрочные контракты в первые 30 дней.
PaymentMethod vs Churn: Chi2: 116.41, P-value: 0.0000
Electronic check показывает наивысший отток среди новых клиентов.
InternetService vs Churn: Chi2: 249.88, P-value: 0.0000
Fiber optic среди новых клиентов имеет критически высокий отток.
Dependents vs Churn: Chi2: 15.75, P-value: 0.0001
Наличие иждивенцев статистически значимо снижает отток.
4. Interaction Analysis: комбинации факторов
Fiber optic + Month-to-month: 74.15% отток (619 клиентов)
Это критический сегмент риска. Почти 3/4 клиентов уходят!
Fiber optic + One year: 100% отток (1 клиент) - малая выборка, но тревожный сигнал.
DSL + Month-to-month: 49.49% отток (495 клиентов)
No internet + Month-to-month: 25.42% отток (299 клиентов)
Вывод: Fiber optic без долгосрочного контракта - рецепт катастрофы.
5. Monthly Charges Threshold Analysis
1234567
Ключевой инсайт:
Клиенты с чеком выше $30 имеют отток 63%+
Клиенты с чеком ниже $30 имеют отток 22-27%
Разница в 3 раза!
Стратегическая рекомендация: Для новых клиентов с чеком $30+ необходима:
Персональная onboarding-программа
Демонстрация ценности в первые 30 дней
Промо-акции на 2-3 месяц для удержания
Proactive customer success менеджмент
6. Service Adoption: Churned vs Retained
Визуализация показывает, что retained клиенты значительно чаще подключают дополнительные услуги в первые месяцы.
Инсайт: Cross-selling в первые 90 дней - не просто revenue driver, а retention tool.
Сводные стратегические рекомендации:
Приоритет 1: Критический период 0-6 месяцев
Внедрить 90-day onboarding program
Еженедельные touchpoints в первый месяц
Персональный customer success manager для premium-сегмента
Приоритет 2: Контрактная стратегия
Агрессивный upsell на 1-2 year контракты в первые 30 дней
Special offers для перехода с month-to-month на long-term
Early termination fee waiver при переходе на long-term
Приоритет 3: Fiber optic retention
Quality assurance program для fiber клиентов
Proactive technical support в первые 90 дней
Speed test monitoring и proactive optimization
Приоритет 4: Payment method optimization
Incentives для перехода на auto-pay (скидка 2-3%)
Education program о benefits auto-pay
Seamless migration process с electronic check
Приоритет 5: Cross-selling strategy
Bundle packages для новых клиентов (3+ услуги со скидкой)
Avoid over-selling в первые 30 дней
Focus on adoption, не на sales
Приоритет 6: Pricing strategy
Special introductory pricing для $30+ сегмента
Value demonstration before price increase
Gradual ramp-up pricing вместо cliff increase
Заключение: Анализ выявил, что основная проблема оттока сконцентрирована в первых 6 месяцах, особенно среди клиентов с fiber optic, month-to-month контрактами и высоким чеком. Приоритет - программа удержания новых клиентов с персонализированным подходом.